# Tutorial: BESS shortlist workflow

Audience:
- Analysts who need a repeatable path from GIS screening to a GB BESS shortlist.

Prerequisites:
- `luminus-py` installed with notebook dependencies.
- A working `luminus-mcp` install on `PATH`.
- An ENTSO-E key configured so the shortlist flow can include screening-level BESS revenue estimates.

Learning goals:
- Run a single shortlist tool that already combines GIS, revenue, and queue context.
- Inspect both the full ranking and the top shortlist slice.
- Keep the analyst handoff compact and skimmable.
- Produce a notebook a portfolio team can read without extra context.


## Outline

1. Start a GIS client that exposes the shortlist flow.
2. Define a compact candidate set.
3. Run the GB BESS shortlist flow.
4. Split the result into the full ranking and the shortlist.
5. Capture the result for review.


In [ ]:
from __future__ import annotations

from luminus import Luminus
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

lum = Luminus(profile="gis", request_timeout=120.0)
lum


## Step 1 - Define the candidate list

The shortlist starts as a small, human-readable table. That makes it easy to track what was screened and why it survived to the next stage.


In [ ]:
candidate_sites = pd.DataFrame(
    [
        {"label": "A", "site_name": "Alpha", "lat": 52.120, "lon": 0.180},
        {"label": "B", "site_name": "Bravo", "lat": 52.080, "lon": 0.220},
        {"label": "C", "site_name": "Charlie", "lat": 52.055, "lon": 0.165},
        {"label": "D", "site_name": "Delta", "lat": 52.100, "lon": 0.140},
        {"label": "E", "site_name": "Echo", "lat": 52.145, "lon": 0.205},
    ]
)

display(candidate_sites)
candidate_sites.to_dict(orient="records")


## Step 2 - Run the shortlist flow

The shortlist tool already layers GIS ranking, screening-level BESS revenue, GB transmission queue context, and SSEN DNO headroom where public SSEN data resolves. Keep the full payload visible so reviewers can see both the ranked list and the shortlist slice.


In [ ]:
shortlist_result = lum.shortlist_bess_sites(
    country="GB",
    sites=candidate_sites.to_dict(orient="records"),
    capacity_mw=25,
    shortlist_size=3,
)
rankings = shortlist_result.to_pandas(data_key="rankings")

display(rankings.head())
rankings.columns.tolist()


## Step 3 - Split the result into analyst views

The full ranking is useful for auditability. The shortlist frame is the compact handoff for a portfolio or origination review.


In [ ]:
shortlist = shortlist_result.to_pandas(data_key="shortlist")

display(rankings[["rank", "label", "verdict", "estimated_annual_revenue_eur", "queue_total_mw_queued", "dno_generation_headroom_mw", "shortlist_score"]])
display(shortlist)


## Step 4 - Keep a compact review frame

Select the columns that matter for the investment conversation: screening outcome, economics, queue burden, SSEN headroom where available, and any data gaps.


In [ ]:
review_columns = [
    "rank",
    "label",
    "verdict",
    "estimated_annual_revenue_eur",
    "queue_total_mw_queued",
    "dno_headroom_site",
    "dno_generation_headroom_mw",
    "dno_generation_rag_status",
    "shortlist_score",
    "data_gaps",
]
review_frame = shortlist[review_columns].copy()
display(review_frame)
review_frame.columns.tolist()


## Exercises

- Add another GB candidate and see whether it survives into the BESS shortlist.
- Swap the shortlist length from three sites to five sites.
- Change `capacity_mw` and see how the revenue component moves the ranking.


## Pitfall

Do not read the queue number as available capacity. It is only a public transmission-register signal, not a connection offer or a DNO headroom view.


## Extension

If you need more diligence, take the shortlist output and run deeper site-by-site work only on the surviving rows.


In [ ]:
lum.close()
